# Chapter 13: Pooling Cross Sections Across Time — Simple Panel Data Methods

Topics: pooled cross sections, difference-in-differences, organizing panel data, first-differenced estimator.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 13.1 Pooled Cross Sections

In [ ]:
df = woo.dataWoo('cps78_85')
print(df.head())
print(f"\nYears in data: {df['year'].unique()}")

# Pool both years, control for year effect
m_pool = smf.ols('np.log(wage) ~ educ + exper + female + C(year)', data=df).fit()
print(m_pool.summary())

## 13.2 Difference-in-Differences

In [ ]:
df = woo.dataWoo('kielmc')

# DiD: effect of incinerator on house prices
# treated = nearinc, post = y81
m_did = smf.ols('rprice ~ nearinc + y81 + nearinc:y81', data=df).fit()
print(m_did.summary())

did_estimate = m_did.params['nearinc:y81']
print(f"\nDiD Estimate: ${did_estimate:.2f}")
print(f"Houses near incinerator fell by ${abs(did_estimate):.2f} relative to control, after 1981")

## DiD Visualisation

In [ ]:
means = df.groupby(['nearinc', 'y81'])['rprice'].mean().reset_index()
means['group'] = means['nearinc'].map({0: 'Control (far)', 1: 'Treated (near)'})
means['period'] = means['y81'].map({0: 'Pre (1978)', 1: 'Post (1981)'})

fig, ax = plt.subplots(figsize=(8, 5))
for group, gdf in means.groupby('group'):
    ax.plot(gdf['period'].values, gdf['rprice'].values, marker='o', lw=2, label=group, markersize=8)

ax.set_ylabel('Mean House Price ($)')
ax.set_title('Difference-in-Differences: House Prices')
ax.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_did_plot.png", dpi=150)
plt.show()

## 13.4 First-Differenced Estimator

In [ ]:
df = woo.dataWoo('crime2')
df['d_crmrte'] = df['crmrte'] - df.groupby('county')['crmrte'].shift(1)
df['d_unem']   = df['unem']   - df.groupby('county')['unem'].shift(1)

m_fd = smf.ols('d_crmrte ~ d_unem', data=df.dropna()).fit()
print("First-Differenced Estimator:")
print(m_fd.summary())